In [74]:
from pandas import DataFrame
from sklearn.datasets import fetch_openml
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,RobustScaler


In [75]:
df=pd.read_csv('../data/processed.csv')

In [76]:
df.shape

(48790, 15)

# Derive columns

In [77]:
import numpy as np

df['capital-gain'] = np.log1p(df['capital-gain'])
df['capital-loss'] = np.log1p(df['capital-loss'])
# Ensure all values are at least 0 before log1p
df['hours-per-week'] = np.log1p(df['hours-per-week'].clip(lower=0))
# Run this right before or after adding your new column
df = df.copy()

# Now perform your calculation
df['net-capital'] = df['capital-gain'] - df['capital-loss']
df['age_bin'] = pd.cut(df['age'],
                       bins=[0, 25, 40, 60, 120],
                       labels=['young', 'adult', 'mid', 'senior'], include_lowest=True)
df['hours_bin'] = pd.cut(df['hours-per-week'],
                         bins=[0, 25, 40, 60, 120],
                         labels=['low', 'normal', 'high', 'very_high'], include_lowest=True)

In [78]:
df

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,income,net-capital,age_bin,hours_bin
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0.000000,0.0,3.713572,United-States,<=50K,0.000000,young,low
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0.000000,0.0,3.931826,United-States,<=50K,0.000000,adult,low
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0.000000,0.0,3.713572,United-States,>50K,0.000000,adult,low
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,8.947546,0.0,3.713572,United-States,>50K,8.947546,mid,low
4,18,Unknown,103497,Some-college,10,Never-married,Unknown,Own-child,White,Female,0.000000,0.0,3.433987,United-States,<=50K,0.000000,young,low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48785,27,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0.000000,0.0,3.663562,United-States,<=50K,0.000000,adult,low
48786,40,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0.000000,0.0,3.713572,United-States,>50K,0.000000,adult,low
48787,58,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0.000000,0.0,3.713572,United-States,<=50K,0.000000,mid,low
48788,22,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0.000000,0.0,3.044522,United-States,<=50K,0.000000,young,low


# 1. One-hot encoding

In [79]:
y=df.iloc[:,14]
y.to_csv('data/target.csv',index=False)

0        <=50K
1        <=50K
2         >50K
3         >50K
4        <=50K
         ...  
48785    <=50K
48786     >50K
48787    <=50K
48788    <=50K
48789     >50K
Name: income, Length: 48790, dtype: str

In [80]:
# Create a list where index 1 is False (skipped) and others are True
cols_to_keep = [i for i in range(df.shape[1]) if i != 14]
df = pd.get_dummies(df.iloc[:,cols_to_keep], drop_first=True)
df

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,net-capital,workclass_Local-gov,workclass_Never-worked,workclass_Private,...,native-country_United-States,native-country_Unknown,native-country_Vietnam,native-country_Yugoslavia,age_bin_adult,age_bin_mid,age_bin_senior,hours_bin_normal,hours_bin_high,hours_bin_very_high
0,25,226802,7,0.000000,0.0,3.713572,0.000000,False,False,True,...,True,False,False,False,False,False,False,False,False,False
1,38,89814,9,0.000000,0.0,3.931826,0.000000,False,False,True,...,True,False,False,False,True,False,False,False,False,False
2,28,336951,12,0.000000,0.0,3.713572,0.000000,True,False,False,...,True,False,False,False,True,False,False,False,False,False
3,44,160323,10,8.947546,0.0,3.713572,8.947546,False,False,True,...,True,False,False,False,False,True,False,False,False,False
4,18,103497,10,0.000000,0.0,3.433987,0.000000,False,False,False,...,True,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48785,27,257302,12,0.000000,0.0,3.663562,0.000000,False,False,True,...,True,False,False,False,True,False,False,False,False,False
48786,40,154374,9,0.000000,0.0,3.713572,0.000000,False,False,True,...,True,False,False,False,True,False,False,False,False,False
48787,58,151910,9,0.000000,0.0,3.713572,0.000000,False,False,True,...,True,False,False,False,False,True,False,False,False,False
48788,22,201490,9,0.000000,0.0,3.044522,0.000000,False,False,True,...,True,False,False,False,False,False,False,False,False,False


In [63]:
df.shape

(48790, 100)

# 3. Transformation

In [81]:
for col in df.columns:
    if str(col).upper().__contains__('INCOME'):
        print(col)

In [82]:
# scaler = StandardScaler()
scaler = RobustScaler()
num_cols=df.columns
df[num_cols] = scaler.fit_transform(df[num_cols])

In [83]:
df

,age,fnlwgt,education-num,capital-gain,capital-loss,hours-per-week,net-capital,workclass_Local-gov,workclass_Never-worked,workclass_Private,...,native-country_United-States,native-country_Unknown,native-country_Vietnam,native-country_Yugoslavia,age_bin_adult,age_bin_mid,age_bin_senior,hours_bin_normal,hours_bin_high,hours_bin_very_high
0,-0.60,0.405356,-1.000000,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.05,-0.735723,-0.333333,0.000000,0.0,1.896714,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,-0.45,1.322873,0.666667,0.000000,0.0,0.000000,0.000000,1.0,0.0,-1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
3,0.35,-0.148399,0.000000,8.947546,0.0,0.000000,8.947546,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,-0.95,-0.621747,0.000000,0.000000,0.0,-2.429708,0.000000,0.0,0.0,-1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
48785,-0.50,0.659414,0.666667,0.000000,0.0,-0.434611,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
48786,0.15,-0.197953,-0.333333,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
48787,1.05,-0.218478,-0.333333,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
48788,-0.75,0.194513,-0.333333,0.000000,0.0,-5.814318,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [84]:
df.to_csv('data/feature-engineered.csv',index=False)